# Example 7.6 / 2: Count Comparisons with atleast

Companion notebook for **7.6 Counting Constraints** in the AMPL lab Documentation, section "Count Comparisons with atleast". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Count Comparisons with atleast

The condition inside a counting operator can compare variables. Here two plants ship to three
customers, and each plant must send at least 5 units to `minserve[i]` of them, so plant `A` cannot
leave its most expensive customer to plant `B`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;                         # plants
set DEST;                         # customers
param supply {ORIG} >= 0;         # units available at each plant
param demand {DEST} >= 0;         # units each customer needs
param cost {ORIG, DEST} >= 0;     # cost per unit shipped
param minload >= 0;              # smallest shipment that counts as serving a customer
param minserve {ORIG} integer >= 0;  # customers each plant must serve

var Ship {ORIG, j in DEST} integer >= 0, <= demand[j];  # whole units shipped

minimize TotalCost: sum {i in ORIG, j in DEST} cost[i,j] * Ship[i,j];
subject to Supply {i in ORIG}: sum {j in DEST} Ship[i,j] <= supply[i];
subject to Demand {j in DEST}: sum {i in ORIG} Ship[i,j] = demand[j];
# each plant sends at least minload to minserve[i] customers
subject to ServeEnough {i in ORIG}:
    atleast minserve[i] {j in DEST} (Ship[i,j] >= minload);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIG"] = ["A", "B"]
ampl.set["DEST"] = ["x", "y", "z"]
ampl.param["supply"] = {"A": 60, "B": 60}
ampl.param["demand"] = {"x": 30, "y": 30, "z": 30}
ampl.param["cost"] = {("A", "x"): 1, ("A", "y"): 2, ("A", "z"): 4,
                      ("B", "x"): 3, ("B", "y"): 2, ("B", "z"): 1}
ampl.param["minload"] = 5
ampl.param["minserve"] = {"A": 3, "B": 1}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.var["Ship"].to_pandas().unstack())

HiGHS 1.14.0: optimal solution; objective 135
3 simplex iterations
1 branching nodes
       Ship.val        
index1        x   y   z
index0                 
A            30  25   5
B             0   5  25


### Pure AMPL: data.dat and model.run

Without `ServeEnough`, plant `A` would ship nothing to `z`, where it costs 4 per unit. The count
makes `A` serve all three customers, and it sends exactly the minimum 5 units to `z`. The
condition is evaluated for every customer and the number of true instances is constrained; the
bounds on `Ship` let MP reformulate each comparison.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ORIG := A B;
set DEST := x y z;
param supply := A 60  B 60;
param demand := x 30  y 30  z 30;
param cost:  x  y  z :=
         A   1  2  4
         B   3  2  1 ;
param minload := 5;
param minserve := A 3  B 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Ship;

HiGHS 1.14.0: optimal solution; objective 135
3 simplex iterations
1 branching nodes
Ship :=
A x   30
A y   25
A z    5
B x    0
B y    5
B z   25
;

